In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
from IPython.display import display

# Load the full image data
df = pd.read_csv('/Users/nogashlomi/projects/Image_data/dissertation/full_image_data_feb_25.csv', low_memory=False)

# Load the books data
books = pd.read_csv('/Users/nogashlomi/projects/Image_data/dissertation/full_book_data_feb_25.csv', low_memory=False)

# Load the visual tags for SA 1.6
visual_tags = pd.read_excel('/Users/nogashlomi/projects/Image_data/visual_tags/VT_1.6_sphericity_earth_clusters.xlsx')

# Merge on cluster_name
df_full = pd.merge(df, visual_tags, on='cluster_name', how='left')

# Drop rows where year or place or custom_identifier might be null
# And also clean the custom_identifier to string for plotting colors
df_clean = df_full.dropna(subset=['year', 'place', 'custom_identifier']).copy()
df_clean['custom_identifier_str'] = df_clean['custom_identifier'].astype(int).astype(str)

# Target CKs from SA_1.6
target_cks = [
    'CK_Sphericity of the Earth',
    'CK_Visibility of Stars from Varied Locations',
    'CK_Rising and Setting Times at Different Locations',
    'CK_Lunar Eclipse'
]

base_ck = 'CK_Sphericity of the Earth'
other_cks = [ck for ck in target_cks if ck != base_ck]

unique_books_count = books.groupby('year_interval', observed=False)['book'].nunique()
intervals = unique_books_count.index.astype(str)

print("Data loaded perfectly.")


In [ ]:
# Calculate the intersection: Books that have base_ck AND another ck
# 1. Get the set of books that have the base_ck for each interval
base_books_by_interval = (
    df[df['cks'] == base_ck]
    .groupby('year_interval', observed=False)['book']
    .apply(set)
    .to_dict()
)

# 2. For each other CK, get the set of books, and find the intersection with base_ck
intersection_counts = {}
all_other_books_by_interval = {interval: set() for interval in intervals}

for ck in other_cks:
    ck_books_by_interval = (
        df[df['cks'] == ck]
        .groupby('year_interval', observed=False)['book']
        .apply(set)
        .to_dict()
    )
    
    interval_intersections = {}
    for interval in intervals:
        base_set = base_books_by_interval.get(interval, set())
        ck_set = ck_books_by_interval.get(interval, set())
        interval_intersections[interval] = len(base_set.intersection(ck_set))
        all_other_books_by_interval[interval].update(ck_set)
        
    intersection_counts[ck] = interval_intersections

only_base_counts = {}
for interval in intervals:
    base_set = base_books_by_interval.get(interval, set())
    other_set = all_other_books_by_interval.get(interval, set())
    only_base_counts[interval] = len(base_set - other_set)

base_counts = {
    interval: len(base_books_by_interval.get(interval, set()))
    for interval in intervals
}

df_intersections = pd.DataFrame(intersection_counts)
df_intersections['Total Sphericity of the Earth'] = pd.Series(base_counts)
df_intersections['ONLY Sphericity of Earth (No Others)'] = pd.Series(only_base_counts)
df_intersections = df_intersections.fillna(0).astype(int)

cols = ['Total Sphericity of the Earth', 'ONLY Sphericity of Earth (No Others)'] + other_cks
df_intersections = df_intersections[cols]

print(f"\nBooks containing '{base_ck}' vs Intersections:")
display(df_intersections)


In [ ]:
# Plotting the Intersections
plt.figure(figsize=(12, 7))

base_vals = [base_counts.get(interval, 0) for interval in intervals]
plt.plot(intervals, base_vals, label=f'Total {base_ck}', marker='D', color='black', linewidth=2, linestyle='-')

only_base_vals = [only_base_counts.get(interval, 0) for interval in intervals]
plt.plot(intervals, only_base_vals, label=f'ONLY {base_ck}', marker='X', color='red', linewidth=2, linestyle='--')

colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, ck in enumerate(other_cks):
    data = intersection_counts[ck]
    vals = [data.get(interval, 0) for interval in intervals]
    
    non_zero_vals = [v for v in vals if v > 0]
    non_zero_intervals = [intervals[j] for j, v in enumerate(vals) if v > 0]
    
    if non_zero_vals:
        plt.plot(non_zero_intervals, non_zero_vals, marker='o', label=f'{base_ck} AND {ck}', color=colors[i])

plt.title(f'Books Containing "{base_ck}" AND Other Keywords Over Time', fontsize=16)
plt.xlabel('Year Interval', fontsize=12)
plt.ylabel('Number of Unique Books', fontsize=12)
plt.xticks(rotation=45)

plt.legend(title='Keyword Combinations', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Scatter Plot for Images containing BOTH Sphericity of the Earth AND Lunar Eclipse
ck1 = 'CK_Sphericity of the Earth'
ck2 = 'CK_Lunar Eclipse'

df_ck1 = df[df['cks'] == ck1].copy()
images_ck1 = set(df_ck1['images'].unique())

df_ck2 = df[df['cks'] == ck2].copy()
images_ck2 = set(df_ck2['images'].unique())

shared_images = images_ck1.intersection(images_ck2)

print(f"\nTotal specific images tagged with BOTH '{ck1}' AND '{ck2}': {len(shared_images)}")

if shared_images:
    df_shared = df_ck1[df_ck1['images'].isin(shared_images)].drop_duplicates(subset=['images']).copy()
    
    df_shared_clean = df_shared.dropna(subset=['year', 'place', 'custom_identifier']).copy()
    df_shared_clean['custom_identifier_str'] = df_shared_clean['custom_identifier'].astype(int).astype(str)
    
    # 1. Static Scatter Plot (Place vs Time)
    plt.figure(figsize=(14, 10))
    sns.scatterplot(
        data=df_shared_clean, x='year', y='place', hue='custom_identifier_str',
        palette='tab20', s=80, alpha=0.7, legend=False 
    )
    plt.title(f'Printings of EXACT IMAGES tagged with BOTH\n"{ck1}" & "{ck2}"\nOver Time and Place (Colored by custom_id)', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Place', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # 2. Interactive Scatter Plot (Place vs Time)
    fig = px.scatter(
        df_shared_clean, x='year', y='place', color='custom_identifier_str',
        hover_data=['part_or_adaption_label', 'images', 'bid'],
        title=f'Interactive Plot: EXACT IMAGES with BOTH "{ck1}" & "{ck2}" (by Year and Place)',
        labels={'custom_identifier_str': 'Custom ID'}
    )
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    fig.update_layout(height=800, hovermode='closest')
    fig.show()

    # 3. Static Scatter Plot (Custom ID vs Time)
    plt.figure(figsize=(14, 10))
    sns.scatterplot(
        data=df_shared_clean, x='year', y='custom_identifier_str', hue='place',
        palette='tab20', s=80, alpha=0.7, legend=False 
    )
    plt.title(f'Printings of EXACT IMAGES tagged with BOTH\n"{ck1}" & "{ck2}"\nOver Time and Custom ID (Colored by Place)', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Custom ID (Text Part)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # 4. Interactive Scatter Plot (Custom ID vs Time)
    fig_id = px.scatter(
        df_shared_clean, x='year', y='custom_identifier_str', color='place',
        hover_data=['part_or_adaption_label', 'images', 'bid'],
        title=f'Interactive Plot: EXACT IMAGES with BOTH "{ck1}" & "{ck2}" (by Year and Custom ID)',
        labels={'custom_identifier_str': 'Custom ID', 'place': 'Place'}
    )
    fig_id.update_traces(marker=dict(size=8, opacity=0.7))
    fig_id.update_layout(height=800, hovermode='closest')
    fig_id.show()


In [ ]:
# Scatter Plot for Books containing ONLY Sphericity of the Earth (and no other target CKs)
books_with_base = set(df[df['cks'] == 'CK_Sphericity of the Earth']['book'])
books_with_others = set(df[df['cks'].isin(other_cks)]['book'])
exclusive_books = books_with_base - books_with_others

print(f"\nTotal books with ONLY 'CK_Sphericity of the Earth' (no other group CKs): {len(exclusive_books)}")

df_exclusive_clean = pd.DataFrame()

if exclusive_books:
    df_exclusive = df[(df['book'].isin(exclusive_books)) & (df['cks'] == 'CK_Sphericity of the Earth')].copy()
    df_exclusive = df_exclusive.drop_duplicates(subset=['images']).copy()
    
    df_exclusive_clean = df_exclusive.dropna(subset=['year', 'place', 'custom_identifier']).copy()
    df_exclusive_clean['custom_identifier_str'] = df_exclusive_clean['custom_identifier'].astype(int).astype(str)
    
    # Static Scatter Plot (Custom ID vs Time)
    plt.figure(figsize=(14, 10))
    sns.scatterplot(
        data=df_exclusive_clean, x='year', y='custom_identifier_str', hue='place',
        palette='tab20', s=80, alpha=0.7, legend=False 
    )
    plt.title('Printings of EXACT IMAGES in books with\nONLY "CK_Sphericity of the Earth"\nOver Time and Custom ID (Colored by Place)', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Custom ID (Text Part)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # Interactive Scatter Plot (Custom ID vs Time)
    fig_id_excl = px.scatter(
        df_exclusive_clean, x='year', y='custom_identifier_str', color='place',
        hover_data=['part_or_adaption_label', 'images', 'bid', 'book'],
        title='Interactive Plot: IMAGES in books with ONLY "CK_Sphericity of the Earth" (by Year and Custom ID)',
        labels={'custom_identifier_str': 'Custom ID', 'place': 'Place'}
    )
    fig_id_excl.update_traces(marker=dict(size=8, opacity=0.7))
    fig_id_excl.update_layout(height=800, hovermode='closest')
    fig_id_excl.show()


In [ ]:
# Scatter Plot for 'Sphericity of the Earth' Images (FROM EXCLUSIVE BOOKS) with 'ad absurdum' == 'yes'
tag_col = 'ad absurdum'

if not df_exclusive_clean.empty and tag_col in df_exclusive_clean.columns:
    df_absurdum = df_exclusive_clean[df_exclusive_clean[tag_col] == 'yes'].copy()
    
    print(f"\nTotal EXCLUSIVE images for '{base_ck}' with '{tag_col}' == 'yes': {len(df_absurdum)}")
    
    if len(df_absurdum) > 0:
        # 1. Static Scatter Plot (Place vs Time)
        plt.figure(figsize=(14, 10))
        sns.scatterplot(
            data=df_absurdum,
            x='year',
            y='place',
            hue='custom_identifier_str',
            palette='tab20',  
            s=80,
            alpha=0.7,
            legend=False 
        )
        plt.title(f'Printings of EXCLUSIVE IMAGES for "{base_ck}"\nwith Visual Tag "{tag_col}" == "yes"\nOver Time and Place (Colored by Custom ID)', fontsize=16)
        plt.xlabel('Year', fontsize=12)
        plt.ylabel('Place', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # 2. Interactive Scatter Plot (Place vs Time)
        fig_absurdum_place = px.scatter(
            df_absurdum,
            x='year',
            y='place',
            color='custom_identifier_str',
            hover_data=['part_or_adaption_label', 'images', 'bid', 'book'],
            title=f'Interactive Plot: EXCLUSIVE IMAGES for "{base_ck}" with Visual Tag "{tag_col}" == "yes" (by Place)',
            labels={'custom_identifier_str': 'Custom ID'}
        )
        fig_absurdum_place.update_traces(marker=dict(size=8, opacity=0.7))
        fig_absurdum_place.update_layout(height=800, hovermode='closest')
        fig_absurdum_place.show()
    
        # 3. Static Scatter Plot (Custom ID vs Time)
        plt.figure(figsize=(14, 10))
        sns.scatterplot(
            data=df_absurdum, x='year', y='custom_identifier_str', hue='place',
            palette='tab20', s=80, alpha=0.7, legend=False 
        )
        plt.title(f'Printings of EXCLUSIVE IMAGES for "{base_ck}"\nwith Visual Tag "{tag_col}" == "yes"\nOver Time and Custom ID (Colored by Place)', fontsize=16)
        plt.xlabel('Year', fontsize=12)
        plt.ylabel('Custom ID (Text Part)', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # 4. Interactive Scatter Plot (Custom ID vs Time)
        fig_absurdum = px.scatter(
            df_absurdum, x='year', y='custom_identifier_str', color='place',
            hover_data=['part_or_adaption_label', 'images', 'bid', 'book'],
            title=f'Interactive Plot: EXCLUSIVE IMAGES for "{base_ck}" with Visual Tag "{tag_col}" == "yes" (by Custom ID)',
            labels={'custom_identifier_str': 'Custom ID', 'place': 'Place'}
        )
        fig_absurdum.update_traces(marker=dict(size=8, opacity=0.7))
        fig_absurdum.update_layout(height=800, hovermode='closest')
        fig_absurdum.show()
else:
    print(f"Column '{tag_col}' not found in the merged data.")


In [ ]:
# Proportional Graph for Content Keywords
# Goal: percentage of all printed books containing these specific keywords over time.
intervals = unique_books_count.index.astype(str)
total_books_vals = unique_books_count.values

book_counts = {
    keyword: (
        df[df['cks'] == keyword]
        .groupby('year_interval', observed=False)['book']
        .nunique()
        .to_dict()
    )
    for keyword in target_cks
}

plt.figure(figsize=(12, 7))

for keyword, data in book_counts.items():
    raw_vals = [data.get(interval, 0) for interval in intervals]
    
    # Percentage formula: (Books with keyword / Total Unique Books for Interval) * 100
    pct_values = [(val / tot * 100) if tot > 0 else 0 for val, tot in zip(raw_vals, total_books_vals)]
    
    non_zero_values = [val for val in pct_values if val > 0]
    non_zero_intervals = [intervals[i] for i, val in enumerate(pct_values) if val > 0]
    
    if non_zero_values:
        plt.plot(non_zero_intervals, non_zero_values, marker='o', label=keyword, linewidth=2)

plt.title('Percentage of Printed Books Containing Selected CKs Over Time', fontsize=16)
plt.xlabel('Year Interval', fontsize=12)
plt.ylabel('Percentage of Unique Books (%)', fontsize=12)
plt.xticks(rotation=45)

plt.legend(title='Content Keywords', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Proportional Graph for the 'ad absurdum' Visual Tag
# Goal: percentage of all printed books containing an image tagged visually as 'ad absurdum'.
tag_col = 'ad absurdum'

if tag_col in df_clean.columns:
    books_with_tag = (
        df_clean[df_clean[tag_col] == 'yes']
        .groupby('year_interval', observed=False)['book']
        .nunique()
        .reindex(intervals, fill_value=0)
    )

    pct_values = [(val / tot * 100) if tot > 0 else 0 for val, tot in zip(books_with_tag.values, total_books_vals)]
    
    non_zero_intervals = [interval for interval, val in zip(intervals, pct_values) if val > 0]
    non_zero_values = [val for val in pct_values if val > 0]

    if non_zero_values:
        plt.figure(figsize=(12, 7))
        plt.plot(non_zero_intervals, non_zero_values, marker='^', color='orange', linewidth=2, label=tag_col)

        plt.title('Percentage of Printed Books Containing the "ad absurdum" Visual Tag Over Time', fontsize=16)
        plt.xlabel('Year Interval', fontsize=12)
        plt.ylabel('Percentage of Unique Books (%)', fontsize=12)
        plt.xticks(rotation=45)

        plt.legend(title='Visual Tag', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()
else:
    print(f"Column '{tag_col}' not found in the visual data.")
